# LLM Financial Signal Extraction & Backtesting — Walkthrough

This notebook walks through the full pipeline step-by-step:

1. **Load** extracted signals from SEC 10-Q filings (pre-computed via Claude Sonnet 4.6)
2. **Explore** signal distributions across 30 tickers and 5 sectors
3. **Visualize** sentiment trends over time
4. **Backtest** signals against forward stock returns
5. **Evaluate** statistical significance with factor neutralization

**Dataset:** 363 filings · 30 tickers · 5 sectors · 2021–2026

In [ ]:
import sys
from pathlib import Path

# Add project root to path
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)

DATA_DIR = Path.cwd().parent / "data" / "processed"

SECTOR_MAP = {
    "AAPL": "Tech", "MSFT": "Tech", "GOOGL": "Tech", "AMZN": "Tech",
    "NVDA": "Tech", "META": "Tech", "CRM": "Tech", "ORCL": "Tech",
    "JPM": "Finance", "GS": "Finance", "BAC": "Finance", "MS": "Finance",
    "WFC": "Finance", "BLK": "Finance", "C": "Finance", "USB": "Finance",
    "JNJ": "Healthcare", "UNH": "Healthcare", "PFE": "Healthcare",
    "MRK": "Healthcare", "ABBV": "Healthcare", "CVS": "Healthcare",
    "WMT": "Consumer", "HD": "Consumer", "NKE": "Consumer", "TGT": "Consumer", "MCD": "Consumer",
    "XOM": "Energy", "CVX": "Energy", "COP": "Energy",
}

TICKERS = list(SECTOR_MAP.keys())
print(f"Tickers: {len(TICKERS)} across {len(set(SECTOR_MAP.values()))} sectors")

## 1. Load Aligned Signals

Each filing has been processed through the pipeline:
- **EDGAR fetcher** downloads raw 10-Q HTML from SEC
- **Text cleaner** strips HTML/XBRL, extracts MD&A section
- **Signal extractor** sends MD&A text to Claude Sonnet 4.6 at temperature=0, returns structured JSON
- **Signal aligner** joins signals with next-trading-day entry prices and 1d/5d/21d forward returns

In [ ]:
# Load all aligned signal data
frames = []
for ticker in TICKERS:
    path = DATA_DIR / f"{ticker}_signals_aligned.parquet"
    if path.exists():
        frames.append(pd.read_parquet(path))

df = pd.concat(frames, ignore_index=True)
df["sector"] = df["ticker"].map(SECTOR_MAP)
df["filing_date"] = pd.to_datetime(df["filing_date"])

print(f"Loaded {len(df)} filings across {df['ticker'].nunique()} tickers")
print(f"Date range: {df['filing_date'].min().date()} to {df['filing_date'].max().date()}")
print(f"\nColumns: {list(df.columns)}")
df.head()

## 2. Signal Distributions

Let's look at the distribution of the key signals Claude extracted from the filings.

In [ ]:
# Sentiment score distribution
fig = make_subplots(rows=1, cols=3, subplot_titles=[
    "Sentiment Score Distribution", "Tone Distribution", "Guidance Direction"
])

fig.add_trace(go.Histogram(x=df["sentiment_score"], nbinsx=30, marker_color="#636EFA", name="sentiment"), row=1, col=1)

tone_counts = df["tone"].value_counts()
fig.add_trace(go.Bar(x=tone_counts.index, y=tone_counts.values, marker_color="#EF553B", name="tone"), row=1, col=2)

guidance_counts = df["guidance_direction"].value_counts()
fig.add_trace(go.Bar(x=guidance_counts.index, y=guidance_counts.values, marker_color="#00CC96", name="guidance"), row=1, col=3)

fig.update_layout(height=400, width=1100, title_text="Signal Distributions Across All Filings", showlegend=False)
fig.show()

## 3. Sentiment Over Time by Sector

How does management sentiment vary across sectors and over time?

In [ ]:
# Average sentiment by sector over time (quarterly)
df["quarter"] = df["filing_date"].dt.to_period("Q").astype(str)
sector_quarterly = df.groupby(["quarter", "sector"])["sentiment_score"].mean().reset_index()

fig = px.line(
    sector_quarterly, x="quarter", y="sentiment_score", color="sector",
    title="Average Sentiment Score by Sector (Quarterly)",
    labels={"sentiment_score": "Avg Sentiment Score", "quarter": "Quarter"},
    markers=True,
)
fig.update_layout(height=500, width=1000, xaxis_tickangle=-45)
fig.show()

## 4. Sentiment vs Forward Returns

The key question: does the LLM-extracted sentiment actually predict future stock price movement?

In [ ]:
# Scatter: sentiment vs forward returns at different horizons
fig = make_subplots(rows=1, cols=3, subplot_titles=["1-Day Returns", "5-Day Returns", "21-Day Returns"])

for i, horizon in enumerate(["fwd_return_1d", "fwd_return_5d", "fwd_return_21d"], 1):
    sub = df.dropna(subset=[horizon, "sentiment_score"])
    fig.add_trace(
        go.Scatter(
            x=sub["sentiment_score"], y=sub[horizon],
            mode="markers", marker=dict(size=5, opacity=0.5),
            text=sub["ticker"], name=horizon,
        ), row=1, col=i,
    )
    # Add trendline
    z = np.polyfit(sub["sentiment_score"], sub[horizon], 1)
    x_line = np.linspace(sub["sentiment_score"].min(), sub["sentiment_score"].max(), 50)
    fig.add_trace(
        go.Scatter(x=x_line, y=np.polyval(z, x_line), mode="lines",
                   line=dict(color="red", width=2), showlegend=False),
        row=1, col=i,
    )

fig.update_layout(height=400, width=1200, title_text="Sentiment Score vs Forward Returns", showlegend=False)
fig.update_xaxes(title_text="Sentiment Score")
fig.update_yaxes(title_text="Forward Return")
fig.show()

## 5. Backtest Results

We run formal statistical tests on each signal:
- **Sharpe Ratio** — risk-adjusted return (annualized)
- **Hit Rate** — % of correct directional predictions
- **t-statistic / p-value** — is the mean return significantly different from zero?
- **Pearson correlation** — linear relationship between signal and returns

In [ ]:
import json
from src.backtesting.engine import run_sentiment_backtest, run_tone_backtest

# Run backtests across all horizons
results_rows = []
for horizon in ["fwd_return_1d", "fwd_return_5d", "fwd_return_21d"]:
    sent = run_sentiment_backtest(df, horizon)
    if sent:
        results_rows.append({
            "Signal": "Sentiment Score",
            "Horizon": horizon.replace("fwd_return_", ""),
            "Sharpe": sent.get("overall_sharpe"),
            "Hit Rate": f"{sent.get('hit_rate', 0):.1%}",
            "t-stat": sent.get("t_stat"),
            "p-value": sent.get("p_value"),
            "Significant": "Yes" if sent.get("significant") else "No",
        })
    tone = run_tone_backtest(df, horizon)
    if tone:
        results_rows.append({
            "Signal": "Tone",
            "Horizon": horizon.replace("fwd_return_", ""),
            "Sharpe": tone.get("overall_sharpe"),
            "Hit Rate": f"{tone.get('hit_rate', 0):.1%}",
            "t-stat": tone.get("t_stat"),
            "p-value": tone.get("p_value"),
            "Significant": "Yes" if tone.get("significant") else "No",
        })

results_df = pd.DataFrame(results_rows)
results_df.style.applymap(
    lambda v: "background-color: #d4edda" if v == "Yes" else "",
    subset=["Significant"]
)

## 6. Factor Neutralization — The Real Test

Raw returns can be misleading. If the market or sector is up, even a random signal looks good. We neutralize by subtracting the sector ETF return over the same horizon:

```
neutralized_return = raw_return - sector_ETF_return
```

Sectors: SPY (market), XLK (tech), XLF (finance), XLV (healthcare), XLP (consumer), XLE (energy)

In [ ]:
from src.backtesting.factor_model import add_neutralized_columns

# Add neutralized return columns
df_neutral = add_neutralized_columns(df)

# Run backtests on neutralized returns
neutral_rows = []
for horizon in ["fwd_return_1d", "fwd_return_5d", "fwd_return_21d"]:
    nh = f"{horizon}_neutralized"
    if nh not in df_neutral.columns:
        continue
    sent = run_sentiment_backtest(df_neutral, nh)
    if sent:
        neutral_rows.append({
            "Signal": "Sentiment (neutralized)",
            "Horizon": horizon.replace("fwd_return_", ""),
            "Sharpe": sent.get("overall_sharpe"),
            "Hit Rate": f"{sent.get('hit_rate', 0):.1%}",
            "t-stat": sent.get("t_stat"),
            "p-value": sent.get("p_value"),
            "Significant": "Yes" if sent.get("significant") else "No",
        })

neutral_df = pd.DataFrame(neutral_rows)
print("After sector factor neutralization:")
neutral_df

## 7. Key Themes Extracted by Claude

Let's look at the most common themes Claude identified across all filings.

In [ ]:
# Flatten key_themes and count
from collections import Counter

all_themes = []
for themes in df["key_themes"].dropna():
    if isinstance(themes, list):
        all_themes.extend([t.lower().strip() for t in themes])
    elif isinstance(themes, str):
        all_themes.append(themes.lower().strip())

theme_counts = Counter(all_themes).most_common(20)
theme_df = pd.DataFrame(theme_counts, columns=["Theme", "Count"])

fig = px.bar(theme_df, x="Count", y="Theme", orientation="h",
             title="Top 20 Themes Extracted from SEC Filings",
             color="Count", color_continuous_scale="Blues")
fig.update_layout(height=600, width=900, yaxis=dict(autorange="reversed"))
fig.show()

## 8. Conclusions

**Raw signal is significant** — LLM-extracted sentiment from 10-Q filings shows statistically significant predictive power at 5-day (p=0.015) and 21-day (p=0.000) horizons.

**Factor neutralization removes significance** — After controlling for sector momentum, the signal's predictive power disappears. This is a common finding in NLP-based alpha research: text signals often proxy for sector-level sentiment rather than firm-specific information.

**What this means:**
- Management language in SEC filings contains *some* predictive information, but it's largely redundant with sector-level signals
- A more targeted approach (earnings call transcripts, guidance-specific language, cross-sectional ranking within sector) may isolate true firm-level alpha
- The pipeline itself is production-ready and can be extended to test these hypotheses

**Next steps:**
- Add earnings call transcript analysis (more timely than 10-Q filings)
- Test cross-sectional signals (rank within sector instead of absolute)
- Expand to full S&P 500 for statistical power
- Compare Claude vs GPT-4o vs open-source models (Llama, Mistral)